# Test of G4 (`04_repository_collaboration_metrics`)

This notebook tests `04_repository_collaboration_metrics` with three small hand-made Silver tables in the schema `github_lakehouse.test_gold_g4`. It runs the real notebook with `%run` and `test_schema = test_gold_g4`, so **production tables are never written**. The last check section proves that.

It has the same sections as the tests of G1 and G2. What is different: G4 reads three Silver tables, so the test has three of them, and the hand-made rows and the expected values are those of G4.

**Before you run it:** `08_gold_audit_table` must have run once, and the three Silver tables `silver.issue_comment_events`, `silver.pull_request_review_events` and `silver.pull_request_review_comment_events` must exist. The test copies the definitions of the four tables. It does not read their rows.

## How to read the output

- Every check section prints a table with one line per check: `PASS` or `FAIL`, the actual value and the expected value.
- A section stops with an error when one of its checks failed.
- **Four cells are EXPECTED to fail.** The text before each of them says so. "Run all" stops at such a cell. Then click the cell **after** it and choose "Run all below". It does no harm if one of these four cells runs more than once.
- **The result of the whole test is the cell "Result of the whole test" near the end.** It prints `TEST PASSED` only when all 13 sections ran and every check passed. If a section failed or was left out, it stops with `TEST FAILED` and names the section.
- The last cell drops the test schema.

## What is tested

| Test of the contract | Where |
|---|---|
| No input | T3, T4: three empty Silver tables. T14, T15: a source date without rows. Both give `NO_INPUT` and leave Gold unchanged |
| Schema | T7: the Gold table has exactly the columns and types of the contract |
| Exact values | T7: hand-made rows give the hand-calculated Gold rows |
| Missing values | T7: a comment without a target, a review without a state, an event without an actor, an event without a name, an event without a date |
| Zero base | T7: the only percentage is the actor coverage. Its base is the number of events, and a row always has at least one. So the test checks the other end: a day whose events all have no actor has the coverage 0 and 0 people |
| Rerun | T8, T9: a second run changes nothing |
| Late event | T10, T11: new Silver rows in all three tables change the right Gold rows, and the other days are not touched |
| Removed event | T12, T13: removed Silver rows lower one Gold row and remove another |
| Failed check | T16 to T18: three failing quality gates leave Gold unchanged, then a recovery run |
| A day that is gone | T19, T20: a run for all dates removes a day that no longer exists in Silver |
| Bad parameter | T21 to T23: an invalid `source_date`, `test_schema` or `run_id` stops the notebook in its parameter step. T24: nothing was written |
| Production safety | T24: no production table changed |

## T1. Setup: test schema and empty tables

The test schema is dropped and created again, so every test run starts clean.

`CREATE TABLE ... LIKE` makes an empty table with the same columns as another table. The three Silver test tables are empty copies of the real Silver tables, and the audit test table is an empty copy of the real audit table.

The Gold test table is not created here. The notebook under test creates it, which also tests its `CREATE TABLE` statement.

In [0]:
TEST_SCHEMA = "test_gold_g4"

TEST_ISSUE_COMMENTS = f"github_lakehouse.{TEST_SCHEMA}.issue_comment_events"
TEST_REVIEWS = f"github_lakehouse.{TEST_SCHEMA}.pull_request_review_events"
TEST_REVIEW_COMMENTS = f"github_lakehouse.{TEST_SCHEMA}.pull_request_review_comment_events"
TEST_GOLD = f"github_lakehouse.{TEST_SCHEMA}.repository_collaboration_metrics"
TEST_STAGING = f"github_lakehouse.{TEST_SCHEMA}.repository_collaboration_metrics_staging"
TEST_AUDIT = f"github_lakehouse.{TEST_SCHEMA}.gold_pipeline_runs"
TEST_COPY = f"github_lakehouse.{TEST_SCHEMA}.gold_rows_after_run_1"

PROD_ISSUE_COMMENTS = "github_lakehouse.silver.issue_comment_events"
PROD_REVIEWS = "github_lakehouse.silver.pull_request_review_events"
PROD_REVIEW_COMMENTS = "github_lakehouse.silver.pull_request_review_comment_events"
PROD_AUDIT = "github_lakehouse.audit.gold_pipeline_runs"
PROD_GOLD = "github_lakehouse.gold.repository_collaboration_metrics"
PROD_STAGING = "github_lakehouse.gold_staging.repository_collaboration_metrics"

# Every time in this test is UTC.
spark.conf.set("spark.sql.session.timeZone", "UTC")

# Every check section writes its number of failed checks into this dictionary.
# The cell "Result of the whole test" reads it at the end.
section_results = {}

if not spark.catalog.tableExists(PROD_AUDIT):
    raise RuntimeError(f"{PROD_AUDIT} does not exist. Run 08_gold_audit_table first.")

if not spark.catalog.tableExists(PROD_ISSUE_COMMENTS):
    raise RuntimeError(f"{PROD_ISSUE_COMMENTS} does not exist. Run the Silver notebook 04_issue_comment_events first.")

if not spark.catalog.tableExists(PROD_REVIEWS):
    raise RuntimeError(f"{PROD_REVIEWS} does not exist. Run the Silver notebook 05_pull_request_review_events first.")

if not spark.catalog.tableExists(PROD_REVIEW_COMMENTS):
    raise RuntimeError(f"{PROD_REVIEW_COMMENTS} does not exist. Run the Silver notebook 06_pull_request_review_comment_events first.")

# Only a schema whose name starts with test_ is ever dropped.
if not TEST_SCHEMA.startswith("test_"):
    raise RuntimeError("Refusing to drop a schema that is not a test schema.")

spark.sql(f"DROP SCHEMA IF EXISTS github_lakehouse.{TEST_SCHEMA} CASCADE")
spark.sql(f"CREATE SCHEMA github_lakehouse.{TEST_SCHEMA}")

spark.sql(f"CREATE TABLE {TEST_ISSUE_COMMENTS} LIKE {PROD_ISSUE_COMMENTS}")
spark.sql(f"CREATE TABLE {TEST_REVIEWS} LIKE {PROD_REVIEWS}")
spark.sql(f"CREATE TABLE {TEST_REVIEW_COMMENTS} LIKE {PROD_REVIEW_COMMENTS}")
spark.sql(f"CREATE TABLE {TEST_AUDIT} LIKE {PROD_AUDIT}")

print("Test schema ready:", TEST_SCHEMA)

## T2. Remember the state of the production tables

The Delta version of a table goes up with every write. The versions are read now and again at the end. If they are the same, the test did not write to production.

The production Gold table and its staging copy may not exist yet. The value `-1` means: the table does not exist.

In [0]:
prod_audit_version_before = spark.sql(
    f"DESCRIBE HISTORY {PROD_AUDIT} LIMIT 1"
).collect()[0]["version"]

if spark.catalog.tableExists(PROD_GOLD):
    prod_gold_version_before = spark.sql(
        f"DESCRIBE HISTORY {PROD_GOLD} LIMIT 1"
    ).collect()[0]["version"]
else:
    prod_gold_version_before = -1

if spark.catalog.tableExists(PROD_STAGING):
    prod_staging_version_before = spark.sql(
        f"DESCRIBE HISTORY {PROD_STAGING} LIMIT 1"
    ).collect()[0]["version"]
else:
    prod_staging_version_before = -1

print("Production audit version  :", prod_audit_version_before)
print("Production Gold version   :", prod_gold_version_before)
print("Production staging version:", prod_staging_version_before)

### A two-line summary of every audit row

An audit row holds two maps: the Silver rows read per table, and the quality numbers. The checks below compare them as text. This view writes each map into one readable line, for example:

`issue_comments=10 reviews=13 review_comments=9`

`repository_days=8 issue_comments=10 on_issues=5 on_prs=3 unclassified=2 reviews=13 state_known=12 approved=5 changes_requested=2 commented=2 dismissed=1 other_state=2 inline=9 total=32 actor_known=24 actors=14`

The view is not a copy. It always reads the audit table as it is at that moment.

In [0]:
audit_summary = spark.sql(f"""
    SELECT
        run_id,
        concat(
            'issue_comments=',     CAST(source_rows_read['issue_comment_events'] AS STRING),
            ' reviews=',           CAST(source_rows_read['pull_request_review_events'] AS STRING),
            ' review_comments=',   CAST(source_rows_read['pull_request_review_comment_events'] AS STRING)
        ) AS source_summary,
        concat(
            'repository_days=',    CAST(quality_metrics['repository_days'] AS STRING),
            ' issue_comments=',    CAST(quality_metrics['issue_comment_events'] AS STRING),
            ' on_issues=',         CAST(quality_metrics['issue_conversation_comment_events'] AS STRING),
            ' on_prs=',            CAST(quality_metrics['pr_conversation_comment_events'] AS STRING),
            ' unclassified=',      CAST(quality_metrics['unclassified_conversation_comment_events'] AS STRING),
            ' reviews=',           CAST(quality_metrics['review_events'] AS STRING),
            ' state_known=',       CAST(quality_metrics['review_state_known_events'] AS STRING),
            ' approved=',          CAST(quality_metrics['review_approved_events'] AS STRING),
            ' changes_requested=', CAST(quality_metrics['review_changes_requested_events'] AS STRING),
            ' commented=',         CAST(quality_metrics['review_commented_events'] AS STRING),
            ' dismissed=',         CAST(quality_metrics['review_dismissed_events'] AS STRING),
            ' other_state=',       CAST(quality_metrics['review_other_state_events'] AS STRING),
            ' inline=',            CAST(quality_metrics['inline_review_comment_events'] AS STRING),
            ' total=',             CAST(quality_metrics['collaboration_events_total'] AS STRING),
            ' actor_known=',       CAST(quality_metrics['actor_known_events'] AS STRING),
            ' actors=',            CAST(quality_metrics['unique_collaborating_actors'] AS STRING)
        ) AS metrics_summary
    FROM {TEST_AUDIT}
""")
audit_summary.createOrReplaceTempView("audit_summary")

## T3. Run on the empty Silver tables

In [0]:
%run ../04_repository_collaboration_metrics $test_schema="test_gold_g4" $source_date="" $run_id="test-dry-run"

## T4. Checks: no input

Three empty Silver tables have no days to rebuild. The run must end as `NO_INPUT`, publish nothing and still write its audit row.

In every check table, `<=>` compares two values and also treats NULL and NULL as equal.

In [0]:
checks = spark.sql(f"""
    SELECT
        check_name,
        actual,
        expected,
        CASE WHEN actual <=> expected THEN 'PASS' ELSE 'FAIL' END AS result
    FROM (
        SELECT '01 the notebook read the test issue comment table' AS check_name,
               '{ISSUE_COMMENT_TABLE}' AS actual, '{TEST_ISSUE_COMMENTS}' AS expected
        UNION ALL
        SELECT '02 the notebook read the test review table', '{REVIEW_TABLE}', '{TEST_REVIEWS}'
        UNION ALL
        SELECT '03 the notebook read the test review comment table', '{REVIEW_COMMENT_TABLE}', '{TEST_REVIEW_COMMENTS}'
        UNION ALL
        SELECT '04 the notebook wrote the test Gold table', '{GOLD_TABLE}', '{TEST_GOLD}'
        UNION ALL
        SELECT '05 the notebook wrote the test staging table', '{STAGING_TABLE}', '{TEST_STAGING}'
        UNION ALL
        SELECT '06 the notebook wrote the test audit table', '{AUDIT_TABLE}', '{TEST_AUDIT}'
        UNION ALL
        SELECT '07 Gold rows', CAST(COUNT(*) AS STRING), '0' FROM {TEST_GOLD}
        UNION ALL
        SELECT '08 audit rows', CAST(COUNT(*) AS STRING), '1' FROM {TEST_AUDIT}
        UNION ALL
        SELECT '09 audit status', MAX(status), 'NO_INPUT'
        FROM {TEST_AUDIT} WHERE run_id = 'test-dry-run'
        UNION ALL
        SELECT '10 audit pipeline_name', MAX(pipeline_name), 'gold_repository_collaboration_metrics'
        FROM {TEST_AUDIT} WHERE run_id = 'test-dry-run'
        UNION ALL
        SELECT '11 audit gold_table', MAX(gold_table), '{TEST_GOLD}'
        FROM {TEST_AUDIT} WHERE run_id = 'test-dry-run'
        UNION ALL
        SELECT '12 audit processing_scope', MAX(processing_scope), 'ALL_DATES'
        FROM {TEST_AUDIT} WHERE run_id = 'test-dry-run'
        UNION ALL
        SELECT '13 audit gold_rows_published', CAST(MAX(gold_rows_published) AS STRING), '0'
        FROM {TEST_AUDIT} WHERE run_id = 'test-dry-run'
        UNION ALL
        SELECT '14 audit checks_passed', CAST(MAX(checks_passed) AS STRING), 'true'
        FROM {TEST_AUDIT} WHERE run_id = 'test-dry-run'
        UNION ALL
        SELECT '15 audit number of affected dates', CAST(MAX(size(affected_dates)) AS STRING), '0'
        FROM {TEST_AUDIT} WHERE run_id = 'test-dry-run'
        UNION ALL
        SELECT '16 audit Silver rows read', MAX(source_summary), 'issue_comments=0 reviews=0 review_comments=0'
        FROM audit_summary WHERE run_id = 'test-dry-run'
        UNION ALL
        SELECT '17 audit repository_days', CAST(MAX(quality_metrics['repository_days']) AS STRING), '0'
        FROM {TEST_AUDIT} WHERE run_id = 'test-dry-run'
    )
    ORDER BY check_name
""")
checks.show(100, truncate=False)

failed_count = checks.filter("result = 'FAIL'").count()
section_results["T4"] = failed_count
if failed_count > 0:
    raise AssertionError(f"T4: {failed_count} check(s) failed. See the table above.")
print("T4: ALL CHECKS PASSED")

## T5. Load the hand-made Silver rows

34 events on three days: 11 issue comments, 14 reviews and 9 inline review comments. Only the columns that matter here are filled, the others stay NULL. The events are numbered ...01 to ...34 across the three tables, so every number exists only once.

**Repository 100 on 1 February: 15 events, one for every rule**

| Event | Table | Actor | What it tests |
|---|---|---|---|
| ...01 | Issue comment | 1 | A comment on an issue |
| ...02 | Issue comment | 1 | A comment in a pull request conversation, by the same person |
| ...03 | Issue comment | 2 | A second comment in a pull request conversation |
| ...04 | Issue comment | 3 | No target and no action: it counts as an issue comment with an unknown target |
| ...05 | Issue comment | none | No actor: it counts as an event, but not as a person |
| ...06 | Review | 2 | `approved`. Actor 2 also commented (...03): one person, not two |
| ...07 | Review | 4 | `changes_requested` |
| ...08 | Review | 4 | `commented`, by the same person |
| ...09 | Review | 5 | `dismissed` |
| ...10 | Review | 5 | `pending`: a known state that is none of the four |
| ...11 | Review | 6 | No state: it counts as a review, but in none of the five state columns |
| ...12 | Review | none | `approved` without an actor. Its name is `org/alpha-old` |
| ...13 | Inline review comment | 1 | Same second as ...12, but the larger `event_id`. Its name is `org/alpha-new` |
| ...14 | Inline review comment | 7 | An earlier inline review comment |
| ...15 | Inline review comment | 7 | The latest event of the day, without a repository name |

**The other rows**

| Repository | Day | Events | What it tests |
|---|---|---|---|
| 200 | 1 Feb | 2 | Only comments on issues: every review column is 0. Actor 1 is also active in repository 100, and counts in both |
| 200 | none | 2 | An issue comment and a review without event time and date: they cannot be placed on a day |
| 200 | 2 Feb | 2 | The same repository on a second day, with a new name. One person comments and reviews: 2 events, 1 person |
| 400 | 2 Feb | 2 | A review and an inline review comment without actor and without name: coverage 0, 0 people, no label. They have no target, but they are not issue comments, so the unknown-target column stays 0 |
| 300 | 5 Feb | 3 | Only reviews. 2 of 3 events have an actor: a coverage that must be rounded |
| 310 | 5 Feb | 4 | Only inline review comments, all by one person: 4 events, 1 person |
| 320 | 5 Feb | 3 | One event of each kind. The review state `APPROVED` is written in capital letters. That is not the state `approved`, so it counts as another state |
| 600 | 5 Feb | 1 | An issue comment with nothing but a repository and a time: unknown target, no actor |

In [0]:
spark.sql("INSERT INTO " + TEST_ISSUE_COMMENTS + """
    (event_id, event_timestamp, event_date, actor_id, repo_id, repo_name,
     comment_action, comment_target_type, _source_date)
VALUES
    -- Repository 100 on 1 February
    ('44000000001', TIMESTAMP'2024-02-01 08:00:00', DATE'2024-02-01', 1,    100, 'org/alpha',        'created', 'issue',        DATE'2024-02-01'),
    ('44000000002', TIMESTAMP'2024-02-01 08:30:00', DATE'2024-02-01', 1,    100, 'org/alpha',        'created', 'pull_request', DATE'2024-02-01'),
    ('44000000003', TIMESTAMP'2024-02-01 09:00:00', DATE'2024-02-01', 2,    100, 'org/alpha',        'created', 'pull_request', DATE'2024-02-01'),
    ('44000000004', TIMESTAMP'2024-02-01 09:30:00', DATE'2024-02-01', 3,    100, 'org/alpha',        NULL,      NULL,           DATE'2024-02-01'),
    ('44000000005', TIMESTAMP'2024-02-01 10:00:00', DATE'2024-02-01', NULL, 100, 'org/alpha',        'created', 'issue',        DATE'2024-02-01'),

    -- Repository 200 on 1 February: only comments on issues
    ('44000000016', TIMESTAMP'2024-02-01 09:00:00', DATE'2024-02-01', 1,    200, 'org/beta',         'created', 'issue',        DATE'2024-02-01'),
    ('44000000017', TIMESTAMP'2024-02-01 09:30:00', DATE'2024-02-01', 8,    200, 'org/beta',         'created', 'issue',        DATE'2024-02-01'),

    -- Repository 200: an event without event time and event date
    ('44000000018', NULL,                           NULL,             8,    200, 'org/beta',         'created', 'issue',        DATE'2024-02-01'),

    -- Repository 200 on 2 February: a new name
    ('44000000020', TIMESTAMP'2024-02-02 09:00:00', DATE'2024-02-02', 1,    200, 'org/beta-renamed', 'created', 'pull_request', DATE'2024-02-02'),

    -- Repository 320 on 5 February
    ('44000000031', TIMESTAMP'2024-02-05 06:00:00', DATE'2024-02-05', 30,   320, 'org/epsilon',      'created', 'issue',        DATE'2024-02-05'),

    -- Repository 600 on 5 February: nothing but a repository and a time
    ('44000000034', TIMESTAMP'2024-02-05 15:00:00', DATE'2024-02-05', NULL, 600, 'org/eta',          NULL,      NULL,           DATE'2024-02-05')
""")

spark.sql("INSERT INTO " + TEST_REVIEWS + """
    (event_id, event_timestamp, event_date, actor_id, repo_id, repo_name,
     review_action, review_state, _source_date)
VALUES
    -- Repository 100 on 1 February
    ('44000000006', TIMESTAMP'2024-02-01 10:30:00', DATE'2024-02-01', 2,    100, 'org/alpha',        'created', 'approved',          DATE'2024-02-01'),
    ('44000000007', TIMESTAMP'2024-02-01 11:00:00', DATE'2024-02-01', 4,    100, 'org/alpha',        'created', 'changes_requested', DATE'2024-02-01'),
    ('44000000008', TIMESTAMP'2024-02-01 11:30:00', DATE'2024-02-01', 4,    100, 'org/alpha',        'created', 'commented',         DATE'2024-02-01'),
    ('44000000009', TIMESTAMP'2024-02-01 12:00:00', DATE'2024-02-01', 5,    100, 'org/alpha',        'created', 'dismissed',         DATE'2024-02-01'),
    ('44000000010', TIMESTAMP'2024-02-01 12:30:00', DATE'2024-02-01', 5,    100, 'org/alpha',        'created', 'pending',           DATE'2024-02-01'),
    ('44000000011', TIMESTAMP'2024-02-01 13:00:00', DATE'2024-02-01', 6,    100, 'org/alpha',        'created', NULL,                DATE'2024-02-01'),
    ('44000000012', TIMESTAMP'2024-02-01 14:00:00', DATE'2024-02-01', NULL, 100, 'org/alpha-old',    'created', 'approved',          DATE'2024-02-01'),

    -- Repository 200: an event without event time and event date
    ('44000000019', NULL,                           NULL,             8,    200, 'org/beta',         'created', 'approved',          DATE'2024-02-01'),

    -- Repository 200 on 2 February: the person who commented also reviews
    ('44000000021', TIMESTAMP'2024-02-02 09:30:00', DATE'2024-02-02', 1,    200, 'org/beta-renamed', 'created', 'approved',          DATE'2024-02-02'),

    -- Repository 400 on 2 February: no actor, no name
    ('44000000022', TIMESTAMP'2024-02-02 10:00:00', DATE'2024-02-02', NULL, 400, NULL,               'created', 'commented',         DATE'2024-02-02'),

    -- Repository 300 on 5 February: only reviews
    ('44000000024', TIMESTAMP'2024-02-05 01:00:00', DATE'2024-02-05', 10,   300, 'org/gamma',        'created', 'approved',          DATE'2024-02-05'),
    ('44000000025', TIMESTAMP'2024-02-05 02:00:00', DATE'2024-02-05', 11,   300, 'org/gamma',        'created', 'approved',          DATE'2024-02-05'),
    ('44000000026', TIMESTAMP'2024-02-05 03:00:00', DATE'2024-02-05', NULL, 300, 'org/gamma',        'created', 'changes_requested', DATE'2024-02-05'),

    -- Repository 320 on 5 February: a state in capital letters
    ('44000000032', TIMESTAMP'2024-02-05 07:00:00', DATE'2024-02-05', NULL, 320, 'org/epsilon',      'created', 'APPROVED',          DATE'2024-02-05')
""")

spark.sql("INSERT INTO " + TEST_REVIEW_COMMENTS + """
    (event_id, event_timestamp, event_date, actor_id, repo_id, repo_name,
     comment_action, _source_date)
VALUES
    -- Repository 100 on 1 February
    ('44000000013', TIMESTAMP'2024-02-01 14:00:00', DATE'2024-02-01', 1,    100, 'org/alpha-new',    'created', DATE'2024-02-01'),
    ('44000000014', TIMESTAMP'2024-02-01 13:30:00', DATE'2024-02-01', 7,    100, 'org/alpha',        'created', DATE'2024-02-01'),
    ('44000000015', TIMESTAMP'2024-02-01 15:00:00', DATE'2024-02-01', 7,    100, NULL,               'created', DATE'2024-02-01'),

    -- Repository 400 on 2 February: no actor, no name
    ('44000000023', TIMESTAMP'2024-02-02 10:30:00', DATE'2024-02-02', NULL, 400, NULL,               'created', DATE'2024-02-02'),

    -- Repository 310 on 5 February: four comments by one person
    ('44000000027', TIMESTAMP'2024-02-05 12:00:00', DATE'2024-02-05', 20,   310, 'org/delta',        'created', DATE'2024-02-05'),
    ('44000000028', TIMESTAMP'2024-02-05 12:10:00', DATE'2024-02-05', 20,   310, 'org/delta',        'created', DATE'2024-02-05'),
    ('44000000029', TIMESTAMP'2024-02-05 12:20:00', DATE'2024-02-05', 20,   310, 'org/delta',        'created', DATE'2024-02-05'),
    ('44000000030', TIMESTAMP'2024-02-05 12:30:00', DATE'2024-02-05', 20,   310, 'org/delta',        'created', DATE'2024-02-05'),

    -- Repository 320 on 5 February
    ('44000000033', TIMESTAMP'2024-02-05 08:00:00', DATE'2024-02-05', NULL, 320, 'org/epsilon',      'created', DATE'2024-02-05')
""")

print("Issue comment test rows :", spark.table(TEST_ISSUE_COMMENTS).count())
print("Review test rows        :", spark.table(TEST_REVIEWS).count())
print("Review comment test rows:", spark.table(TEST_REVIEW_COMMENTS).count())

### A two-line summary of every Gold row

The checks below compare whole Gold rows. This view writes the value columns of a row into two readable lines, for example:

`issue_comments=2 on_issues=2 on_prs=0 unclassified=0 reviews=0 state_known=0 approved=0 changes_requested=0 commented=0 dismissed=0 other_state=0 inline=0 total=2`

`actor_known=2 actors=2 coverage=100.0 name=org/beta`

The first line holds the numbers that count events. The second line holds the people, the coverage and the name.

The view is not a copy. It always reads the Gold table as it is at that moment.

In [0]:
gold_summary = spark.sql(f"""
    SELECT
        repo_id,
        activity_date,
        concat(
            'issue_comments=',     CAST(issue_comment_events AS STRING),
            ' on_issues=',         CAST(issue_conversation_comment_events AS STRING),
            ' on_prs=',            CAST(pr_conversation_comment_events AS STRING),
            ' unclassified=',      CAST(unclassified_conversation_comment_events AS STRING),
            ' reviews=',           CAST(review_events AS STRING),
            ' state_known=',       CAST(review_state_known_events AS STRING),
            ' approved=',          CAST(review_approved_events AS STRING),
            ' changes_requested=', CAST(review_changes_requested_events AS STRING),
            ' commented=',         CAST(review_commented_events AS STRING),
            ' dismissed=',         CAST(review_dismissed_events AS STRING),
            ' other_state=',       CAST(review_other_state_events AS STRING),
            ' inline=',            CAST(inline_review_comment_events AS STRING),
            ' total=',             CAST(collaboration_events_total AS STRING)
        ) AS event_summary,
        concat(
            'actor_known=',        CAST(actor_known_events AS STRING),
            ' actors=',            CAST(unique_collaborating_actors AS STRING),
            ' coverage=',          COALESCE(CAST(collaborator_actor_coverage_pct AS STRING), 'NULL'),
            ' name=',              COALESCE(repo_name, 'NULL')
        ) AS people_summary
    FROM {TEST_GOLD}
""")
gold_summary.createOrReplaceTempView("gold_summary")

## T6. Run 1: all dates

In [0]:
%run ../04_repository_collaboration_metrics $test_schema="test_gold_g4" $source_date="" $run_id="test-run1"

## T7. Checks of run 1: schema and exact values

Worked out by hand.

**The run.** 32 of the 34 rows have an event date, on 3 days: 10 issue comments, 13 reviews and 9 inline review comments. The two rows without a date are left out and counted. The 32 rows belong to 8 repository-days.

**Repository 100 on 1 February: the events.**

- 5 issue comments: 2 on issues (...01, ...05), 2 in pull request conversations (...02, ...03) and 1 with an unknown target (...04). 2 + 2 + 1 = 5.
- 7 reviews. 6 have a known state, all but ...11: 2 approved (...06, ...12), 1 changes requested, 1 commented, 1 dismissed and 1 other (`pending`). 2 + 1 + 1 + 1 + 1 = 6.
- 3 inline review comments.
- Total: 5 + 7 + 3 = 15.

**Repository 100 on 1 February: the people.**

- 13 of the 15 events have an actor. ...05 and ...12 have none.
- The actors per kind: issue comments 1, 2 and 3. Reviews 2, 4, 5 and 6. Inline review comments 1 and 7. Counted per kind that would be 3 + 4 + 2 = 9. Counted once across the three kinds it is 7 people: 1, 2, 3, 4, 5, 6 and 7.
- Coverage: 100 × 13 ÷ 15 = 86.6667, rounded 86.67.
- Label: the latest event (...15) has no name, so it is skipped. The next two share the same second: the review ...12 and the inline review comment ...13. The larger `event_id` is ...13, so the label is `org/alpha-new`. The label comes from the latest event of all three tables together.

**The other rows.**

- Repository 200 on 1 February: 2 comments on issues by the actors 1 and 8. Every review column and the inline column are 0, not NULL. 2 people, coverage 100.
- Repository 200 on 2 February: one comment in a pull request conversation and one approving review, both by actor 1. 2 events, 2 with an actor, 1 person. Its label is the new name.
- Repository 400 on 2 February: one review (`commented`) and one inline review comment, both without actor and name. 0 events with an actor, 0 people, and the coverage is 0. The label is NULL. No issue comment, so all four issue comment columns are 0.
- Repository 300 on 5 February: 3 reviews, 2 approved and 1 changes requested. 2 of the 3 events have an actor: 100 × 2 ÷ 3 = 66.6667, rounded 66.67. 2 people.
- Repository 310 on 5 February: 4 inline review comments by actor 20. 4 events, 1 person.
- Repository 320 on 5 February: 1 comment on an issue, 1 review and 1 inline review comment. The state `APPROVED` is known, but it is none of the four states, so it is counted as other. Only the issue comment has an actor: 100 × 1 ÷ 3 = 33.3333, rounded 33.33.
- Repository 600 on 5 February: 1 issue comment with an unknown target and no actor.

**The totals of the run.**

- 10 issue comments: 5 on issues, 3 in pull request conversations, 2 with an unknown target.
- 13 reviews, 12 with a known state: 5 approved, 2 changes requested, 2 commented, 1 dismissed, 2 other.
- 9 inline review comments. Total 32.
- 8 events have no actor, so 24 have one.
- People per row: 7 + 2 + 1 + 0 + 2 + 1 + 1 + 0 = 14.

In [0]:
actual_schema = spark.table(TEST_GOLD).schema.simpleString()
staging_schema = spark.table(TEST_STAGING).schema.simpleString()

expected_schema = (
    "struct<repo_id:bigint,activity_date:date,repo_name:string,"
    "issue_comment_events:bigint,issue_conversation_comment_events:bigint,"
    "pr_conversation_comment_events:bigint,unclassified_conversation_comment_events:bigint,"
    "review_events:bigint,review_state_known_events:bigint,review_approved_events:bigint,"
    "review_changes_requested_events:bigint,review_commented_events:bigint,"
    "review_dismissed_events:bigint,review_other_state_events:bigint,"
    "inline_review_comment_events:bigint,"
    "collaboration_events_total:bigint,actor_known_events:bigint,"
    "unique_collaborating_actors:bigint,collaborator_actor_coverage_pct:double,"
    "_gold_run_id:string,_gold_updated_at:timestamp>"
)

print("Gold    :", actual_schema)
print("Staging :", staging_schema)
print("Expected:", expected_schema)

# The three texts are long. The check table below gets a short text for each comparison.
if actual_schema == expected_schema:
    gold_schema_result = "as in the contract"
else:
    gold_schema_result = "DIFFERENT, see the lines above"

if staging_schema == expected_schema:
    staging_schema_result = "as in the contract"
else:
    staging_schema_result = "DIFFERENT, see the lines above"

In [0]:
checks = spark.sql(f"""
    SELECT
        check_name,
        actual,
        expected,
        CASE WHEN actual <=> expected THEN 'PASS' ELSE 'FAIL' END AS result
    FROM (
        SELECT '01 columns and types of the Gold table' AS check_name,
               '{gold_schema_result}' AS actual, 'as in the contract' AS expected
        UNION ALL
        SELECT '02 columns and types of the staging table', '{staging_schema_result}', 'as in the contract'
        UNION ALL
        SELECT '03 Gold rows', CAST(COUNT(*) AS STRING), '8' FROM {TEST_GOLD}
        UNION ALL
        SELECT '04 different days', CAST(COUNT(DISTINCT activity_date) AS STRING), '3' FROM {TEST_GOLD}
        UNION ALL
        SELECT '05 rows written by this run', CAST(count_if(_gold_run_id = 'test-run1' AND _gold_updated_at IS NOT NULL) AS STRING), '8' FROM {TEST_GOLD}
        UNION ALL
        SELECT '06 sum of collaboration_events_total', CAST(SUM(collaboration_events_total) AS STRING), '32' FROM {TEST_GOLD}
        UNION ALL
        SELECT '07 repository 100 on 1 February: events', MAX(event_summary), 'issue_comments=5 on_issues=2 on_prs=2 unclassified=1 reviews=7 state_known=6 approved=2 changes_requested=1 commented=1 dismissed=1 other_state=1 inline=3 total=15'
        FROM gold_summary WHERE repo_id = 100 AND activity_date = DATE'2024-02-01'
        UNION ALL
        SELECT '08 repository 100 on 1 February: people', MAX(people_summary), 'actor_known=13 actors=7 coverage=86.67 name=org/alpha-new'
        FROM gold_summary WHERE repo_id = 100 AND activity_date = DATE'2024-02-01'
        UNION ALL
        SELECT '09 repository 200 on 1 February (only issue comments): events', MAX(event_summary), 'issue_comments=2 on_issues=2 on_prs=0 unclassified=0 reviews=0 state_known=0 approved=0 changes_requested=0 commented=0 dismissed=0 other_state=0 inline=0 total=2'
        FROM gold_summary WHERE repo_id = 200 AND activity_date = DATE'2024-02-01'
        UNION ALL
        SELECT '10 repository 200 on 1 February (only issue comments): people', MAX(people_summary), 'actor_known=2 actors=2 coverage=100.0 name=org/beta'
        FROM gold_summary WHERE repo_id = 200 AND activity_date = DATE'2024-02-01'
        UNION ALL
        SELECT '11 repository 200 on 2 February (one person, two kinds): events', MAX(event_summary), 'issue_comments=1 on_issues=0 on_prs=1 unclassified=0 reviews=1 state_known=1 approved=1 changes_requested=0 commented=0 dismissed=0 other_state=0 inline=0 total=2'
        FROM gold_summary WHERE repo_id = 200 AND activity_date = DATE'2024-02-02'
        UNION ALL
        SELECT '12 repository 200 on 2 February (one person, two kinds): people', MAX(people_summary), 'actor_known=2 actors=1 coverage=100.0 name=org/beta-renamed'
        FROM gold_summary WHERE repo_id = 200 AND activity_date = DATE'2024-02-02'
        UNION ALL
        SELECT '13 repository 400 on 2 February (no actor, no name): events', MAX(event_summary), 'issue_comments=0 on_issues=0 on_prs=0 unclassified=0 reviews=1 state_known=1 approved=0 changes_requested=0 commented=1 dismissed=0 other_state=0 inline=1 total=2'
        FROM gold_summary WHERE repo_id = 400 AND activity_date = DATE'2024-02-02'
        UNION ALL
        SELECT '14 repository 400 on 2 February (no actor, no name): people', MAX(people_summary), 'actor_known=0 actors=0 coverage=0.0 name=NULL'
        FROM gold_summary WHERE repo_id = 400 AND activity_date = DATE'2024-02-02'
        UNION ALL
        SELECT '15 repository 300 on 5 February (only reviews): events', MAX(event_summary), 'issue_comments=0 on_issues=0 on_prs=0 unclassified=0 reviews=3 state_known=3 approved=2 changes_requested=1 commented=0 dismissed=0 other_state=0 inline=0 total=3'
        FROM gold_summary WHERE repo_id = 300 AND activity_date = DATE'2024-02-05'
        UNION ALL
        SELECT '16 repository 300 on 5 February (only reviews): people', MAX(people_summary), 'actor_known=2 actors=2 coverage=66.67 name=org/gamma'
        FROM gold_summary WHERE repo_id = 300 AND activity_date = DATE'2024-02-05'
        UNION ALL
        SELECT '17 repository 310 on 5 February (one person, four comments): events', MAX(event_summary), 'issue_comments=0 on_issues=0 on_prs=0 unclassified=0 reviews=0 state_known=0 approved=0 changes_requested=0 commented=0 dismissed=0 other_state=0 inline=4 total=4'
        FROM gold_summary WHERE repo_id = 310 AND activity_date = DATE'2024-02-05'
        UNION ALL
        SELECT '18 repository 310 on 5 February (one person, four comments): people', MAX(people_summary), 'actor_known=4 actors=1 coverage=100.0 name=org/delta'
        FROM gold_summary WHERE repo_id = 310 AND activity_date = DATE'2024-02-05'
        UNION ALL
        SELECT '19 repository 320 on 5 February (state in capitals): events', MAX(event_summary), 'issue_comments=1 on_issues=1 on_prs=0 unclassified=0 reviews=1 state_known=1 approved=0 changes_requested=0 commented=0 dismissed=0 other_state=1 inline=1 total=3'
        FROM gold_summary WHERE repo_id = 320 AND activity_date = DATE'2024-02-05'
        UNION ALL
        SELECT '20 repository 320 on 5 February (state in capitals): people', MAX(people_summary), 'actor_known=1 actors=1 coverage=33.33 name=org/epsilon'
        FROM gold_summary WHERE repo_id = 320 AND activity_date = DATE'2024-02-05'
        UNION ALL
        SELECT '21 repository 600 on 5 February (unknown target): events', MAX(event_summary), 'issue_comments=1 on_issues=0 on_prs=0 unclassified=1 reviews=0 state_known=0 approved=0 changes_requested=0 commented=0 dismissed=0 other_state=0 inline=0 total=1'
        FROM gold_summary WHERE repo_id = 600 AND activity_date = DATE'2024-02-05'
        UNION ALL
        SELECT '22 repository 600 on 5 February (unknown target): people', MAX(people_summary), 'actor_known=0 actors=0 coverage=0.0 name=org/eta'
        FROM gold_summary WHERE repo_id = 600 AND activity_date = DATE'2024-02-05'
        UNION ALL
        SELECT '23 audit status', MAX(status), 'SUCCEEDED'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run1'
        UNION ALL
        SELECT '24 audit processing_scope', MAX(processing_scope), 'ALL_DATES'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run1'
        UNION ALL
        SELECT '25 audit source_date', CAST(MAX(source_date) AS STRING), CAST(NULL AS STRING)
        FROM {TEST_AUDIT} WHERE run_id = 'test-run1'
        UNION ALL
        SELECT '26 audit affected_dates', MAX(CAST(affected_dates AS STRING)), '[2024-02-01, 2024-02-02, 2024-02-05]'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run1'
        UNION ALL
        SELECT '27 audit Silver rows read', MAX(source_summary), 'issue_comments=10 reviews=13 review_comments=9'
        FROM audit_summary WHERE run_id = 'test-run1'
        UNION ALL
        SELECT '28 audit rows without an event date', CAST(MAX(rows_excluded['missing_event_date']) AS STRING), '2'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run1'
        UNION ALL
        SELECT '29 audit gold_rows_published', CAST(MAX(gold_rows_published) AS STRING), '8'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run1'
        UNION ALL
        SELECT '30 audit checks_passed', CAST(MAX(checks_passed) AS STRING), 'true'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run1'
        UNION ALL
        SELECT '31 audit failed_checks', MAX(failed_checks), CAST(NULL AS STRING)
        FROM {TEST_AUDIT} WHERE run_id = 'test-run1'
        UNION ALL
        SELECT '32 audit error_message', MAX(error_message), CAST(NULL AS STRING)
        FROM {TEST_AUDIT} WHERE run_id = 'test-run1'
        UNION ALL
        SELECT '33 audit has a table version', CAST(MAX(gold_table_version) IS NOT NULL AS STRING), 'true'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run1'
        UNION ALL
        SELECT '34 audit pipeline_name', MAX(pipeline_name), 'gold_repository_collaboration_metrics'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run1'
        UNION ALL
        SELECT '35 audit gold_table', MAX(gold_table), '{TEST_GOLD}'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run1'
        UNION ALL
        SELECT '36 audit run times are in order', CAST(MAX(run_finished_at >= run_started_at) AS STRING), 'true'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run1'
        UNION ALL
        SELECT '37 audit quality_metrics', MAX(metrics_summary), 'repository_days=8 issue_comments=10 on_issues=5 on_prs=3 unclassified=2 reviews=13 state_known=12 approved=5 changes_requested=2 commented=2 dismissed=1 other_state=2 inline=9 total=32 actor_known=24 actors=14'
        FROM audit_summary WHERE run_id = 'test-run1'
    )
    ORDER BY check_name
""")
checks.show(100, truncate=False)

failed_count = checks.filter("result = 'FAIL'").count()
section_results["T7"] = failed_count
if failed_count > 0:
    raise AssertionError(f"T7: {failed_count} check(s) failed. See the table above.")
print("T7: ALL CHECKS PASSED")

## T8. Run 2: the same run again

First the Gold rows of run 1 are saved as a copy, so that run 2 can be compared with it. The copy holds the key and the two summary lines of every row.

In [0]:
spark.sql(f"""
    CREATE OR REPLACE TABLE {TEST_COPY} AS
    SELECT repo_id, activity_date, event_summary, people_summary
    FROM gold_summary
""")

print("Copy of the Gold rows after run 1:", TEST_COPY)

In [0]:
%run ../04_repository_collaboration_metrics $test_schema="test_gold_g4" $source_date="" $run_id="test-run2"

## T9. Checks of run 2: nothing changed

`EXCEPT` returns the rows of the first query that are not in the second one. Both directions must return no rows. The two technical columns are not part of the summary, because they are new in every run.

A run for all dates replaces the whole table, so every row now carries the ID of run 2.

In [0]:
checks = spark.sql(f"""
    SELECT
        check_name,
        actual,
        expected,
        CASE WHEN actual <=> expected THEN 'PASS' ELSE 'FAIL' END AS result
    FROM (
        SELECT '01 audit status' AS check_name, MAX(status) AS actual, 'SUCCEEDED' AS expected
        FROM {TEST_AUDIT} WHERE run_id = 'test-run2'
        UNION ALL
        SELECT '02 Gold rows', CAST(COUNT(*) AS STRING), '8' FROM {TEST_GOLD}
        UNION ALL
        SELECT '03 rows of run 2 that are not in run 1', CAST(COUNT(*) AS STRING), '0'
        FROM (
            SELECT repo_id, activity_date, event_summary, people_summary FROM gold_summary
            EXCEPT
            SELECT repo_id, activity_date, event_summary, people_summary FROM {TEST_COPY}
        )
        UNION ALL
        SELECT '04 rows of run 1 that are not in run 2', CAST(COUNT(*) AS STRING), '0'
        FROM (
            SELECT repo_id, activity_date, event_summary, people_summary FROM {TEST_COPY}
            EXCEPT
            SELECT repo_id, activity_date, event_summary, people_summary FROM gold_summary
        )
        UNION ALL
        SELECT '05 rows written by run 2', CAST(count_if(_gold_run_id = 'test-run2') AS STRING), '8' FROM {TEST_GOLD}
        UNION ALL
        SELECT '06 audit gold_rows_published', CAST(MAX(gold_rows_published) AS STRING), '8'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run2'
        UNION ALL
        SELECT '07 audit quality_metrics', MAX(metrics_summary), 'repository_days=8 issue_comments=10 on_issues=5 on_prs=3 unclassified=2 reviews=13 state_known=12 approved=5 changes_requested=2 commented=2 dismissed=1 other_state=2 inline=9 total=32 actor_known=24 actors=14'
        FROM audit_summary WHERE run_id = 'test-run2'
        UNION ALL
        SELECT '08 different runs in the audit table so far', CAST(COUNT(DISTINCT run_id) AS STRING), '3' FROM {TEST_AUDIT}
    )
    ORDER BY check_name
""")
checks.show(100, truncate=False)

failed_count = checks.filter("result = 'FAIL'").count()
section_results["T9"] = failed_count
if failed_count > 0:
    raise AssertionError(f"T9: {failed_count} check(s) failed. See the table above.")
print("T9: ALL CHECKS PASSED")

## T10. Run 3: late events, for one source date

Three new Silver rows arrive with the source date 2 February, one in each table:

| Event | Table | Repository | Event date | What it tests |
|---|---|---|---|---|
| ...35 | Issue comment | 200 | 2 Feb | A comment on an issue, on a day that already has a Gold row. Actor 9 is new on that day |
| ...36 | Review | 500 | 2 Feb | A repository that had no Gold row yet |
| ...37 | Inline review comment | 200 | **1 Feb** | A late event: it was loaded with the source date 2 February, but it happened on 1 February. Only the review comment table says that 1 February has to be rebuilt. Actor 8 already commented on that day |

The run gets `source_date = 2024-02-02`.

In [0]:
spark.sql("INSERT INTO " + TEST_ISSUE_COMMENTS + """
    (event_id, event_timestamp, event_date, actor_id, repo_id, repo_name,
     comment_action, comment_target_type, _source_date)
VALUES
    ('44000000035', TIMESTAMP'2024-02-02 11:30:00', DATE'2024-02-02', 9, 200, 'org/beta-renamed', 'created', 'issue', DATE'2024-02-02')
""")

spark.sql("INSERT INTO " + TEST_REVIEWS + """
    (event_id, event_timestamp, event_date, actor_id, repo_id, repo_name,
     review_action, review_state, _source_date)
VALUES
    ('44000000036', TIMESTAMP'2024-02-02 12:00:00', DATE'2024-02-02', 40, 500, 'org/zeta', 'created', 'approved', DATE'2024-02-02')
""")

spark.sql("INSERT INTO " + TEST_REVIEW_COMMENTS + """
    (event_id, event_timestamp, event_date, actor_id, repo_id, repo_name,
     comment_action, _source_date)
VALUES
    ('44000000037', TIMESTAMP'2024-02-01 23:59:00', DATE'2024-02-01', 8, 200, 'org/beta', 'created', DATE'2024-02-02')
""")

print("Issue comment test rows :", spark.table(TEST_ISSUE_COMMENTS).count())
print("Review test rows        :", spark.table(TEST_REVIEWS).count())
print("Review comment test rows:", spark.table(TEST_REVIEW_COMMENTS).count())

In [0]:
%run ../04_repository_collaboration_metrics $test_schema="test_gold_g4" $source_date="2024-02-02" $run_id="test-run3"

## T11. Checks of run 3

Worked out by hand.

- The source date 2 February has 7 rows now: the issue comments ...20 and ...35, the reviews ...21, ...22 and ...36, and the inline review comments ...23 and ...37. Their event dates are 1 and 2 February, so **two** days are affected. The two rows without a date belong to the source date 1 February and are not in the scope of this run.
- Both days are rebuilt from **all** their rows: 18 rows on 1 February (15 + 2 + the late one) and 6 rows on 2 February. 24 rows, 5 repository-days. By table: 9 issue comments, 10 reviews and 5 inline review comments.
- Repository 200 on 1 February has 3 events now: 2 comments on issues and 1 inline review comment. The late event is by actor 8, who already commented that day, so there are still 2 people.
- Repository 200 on 2 February has 3 events: a comment in a pull request conversation, a comment on an issue and an approving review. 2 people: the actors 1 and 9.
- Repository 500 on 2 February is new: one approving review.
- Repository 100 on 1 February and repository 400 on 2 February have the same numbers as before, but they were written again by this run.
- The four rows of 5 February are not touched. They still carry the ID of run 2.
- Gold has 5 + 4 = 9 rows.
- Totals of the run: 9 issue comments (5 on issues, 3 in pull request conversations, 1 unknown). 10 reviews, 9 with a known state: 4 approved, 1 changes requested, 2 commented, 1 dismissed, 1 other. 5 inline review comments. Total 24. 4 events have no actor, so 20 have one. People per row: 7 + 2 + 2 + 0 + 1 = 12.

In [0]:
checks = spark.sql(f"""
    SELECT
        check_name,
        actual,
        expected,
        CASE WHEN actual <=> expected THEN 'PASS' ELSE 'FAIL' END AS result
    FROM (
        SELECT '01 audit status' AS check_name, MAX(status) AS actual, 'SUCCEEDED' AS expected
        FROM {TEST_AUDIT} WHERE run_id = 'test-run3'
        UNION ALL
        SELECT '02 audit processing_scope', MAX(processing_scope), 'SINGLE_DATE'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run3'
        UNION ALL
        SELECT '03 audit source_date', CAST(MAX(source_date) AS STRING), '2024-02-02'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run3'
        UNION ALL
        SELECT '04 audit affected_dates', MAX(CAST(affected_dates AS STRING)), '[2024-02-01, 2024-02-02]'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run3'
        UNION ALL
        SELECT '05 audit Silver rows read', MAX(source_summary), 'issue_comments=9 reviews=10 review_comments=5'
        FROM audit_summary WHERE run_id = 'test-run3'
        UNION ALL
        SELECT '06 audit rows without an event date', CAST(MAX(rows_excluded['missing_event_date']) AS STRING), '0'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run3'
        UNION ALL
        SELECT '07 audit gold_rows_published', CAST(MAX(gold_rows_published) AS STRING), '5'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run3'
        UNION ALL
        SELECT '08 Gold rows', CAST(COUNT(*) AS STRING), '9' FROM {TEST_GOLD}
        UNION ALL
        SELECT '09 rows of 1 and 2 February written by run 3', CAST(count_if(activity_date <= DATE'2024-02-02' AND _gold_run_id = 'test-run3') AS STRING), '5' FROM {TEST_GOLD}
        UNION ALL
        SELECT '10 rows of 5 February still from run 2', CAST(count_if(activity_date = DATE'2024-02-05' AND _gold_run_id = 'test-run2') AS STRING), '4' FROM {TEST_GOLD}
        UNION ALL
        SELECT '11 repository 200 on 1 February (late event): events', MAX(event_summary), 'issue_comments=2 on_issues=2 on_prs=0 unclassified=0 reviews=0 state_known=0 approved=0 changes_requested=0 commented=0 dismissed=0 other_state=0 inline=1 total=3'
        FROM gold_summary WHERE repo_id = 200 AND activity_date = DATE'2024-02-01'
        UNION ALL
        SELECT '12 repository 200 on 1 February (late event): people', MAX(people_summary), 'actor_known=3 actors=2 coverage=100.0 name=org/beta'
        FROM gold_summary WHERE repo_id = 200 AND activity_date = DATE'2024-02-01'
        UNION ALL
        SELECT '13 repository 200 on 2 February (second comment): events', MAX(event_summary), 'issue_comments=2 on_issues=1 on_prs=1 unclassified=0 reviews=1 state_known=1 approved=1 changes_requested=0 commented=0 dismissed=0 other_state=0 inline=0 total=3'
        FROM gold_summary WHERE repo_id = 200 AND activity_date = DATE'2024-02-02'
        UNION ALL
        SELECT '14 repository 200 on 2 February (second comment): people', MAX(people_summary), 'actor_known=3 actors=2 coverage=100.0 name=org/beta-renamed'
        FROM gold_summary WHERE repo_id = 200 AND activity_date = DATE'2024-02-02'
        UNION ALL
        SELECT '15 repository 500 on 2 February (new row): events', MAX(event_summary), 'issue_comments=0 on_issues=0 on_prs=0 unclassified=0 reviews=1 state_known=1 approved=1 changes_requested=0 commented=0 dismissed=0 other_state=0 inline=0 total=1'
        FROM gold_summary WHERE repo_id = 500 AND activity_date = DATE'2024-02-02'
        UNION ALL
        SELECT '16 repository 500 on 2 February (new row): people', MAX(people_summary), 'actor_known=1 actors=1 coverage=100.0 name=org/zeta'
        FROM gold_summary WHERE repo_id = 500 AND activity_date = DATE'2024-02-02'
        UNION ALL
        SELECT '17 repository 100 on 1 February (same numbers): events', MAX(event_summary), 'issue_comments=5 on_issues=2 on_prs=2 unclassified=1 reviews=7 state_known=6 approved=2 changes_requested=1 commented=1 dismissed=1 other_state=1 inline=3 total=15'
        FROM gold_summary WHERE repo_id = 100 AND activity_date = DATE'2024-02-01'
        UNION ALL
        SELECT '18 repository 100 on 1 February (same numbers): people', MAX(people_summary), 'actor_known=13 actors=7 coverage=86.67 name=org/alpha-new'
        FROM gold_summary WHERE repo_id = 100 AND activity_date = DATE'2024-02-01'
        UNION ALL
        SELECT '19 repository 400 on 2 February (same numbers): events', MAX(event_summary), 'issue_comments=0 on_issues=0 on_prs=0 unclassified=0 reviews=1 state_known=1 approved=0 changes_requested=0 commented=1 dismissed=0 other_state=0 inline=1 total=2'
        FROM gold_summary WHERE repo_id = 400 AND activity_date = DATE'2024-02-02'
        UNION ALL
        SELECT '20 repository 400 on 2 February (same numbers): people', MAX(people_summary), 'actor_known=0 actors=0 coverage=0.0 name=NULL'
        FROM gold_summary WHERE repo_id = 400 AND activity_date = DATE'2024-02-02'
        UNION ALL
        SELECT '21 repository 300 on 5 February (not touched): events', MAX(event_summary), 'issue_comments=0 on_issues=0 on_prs=0 unclassified=0 reviews=3 state_known=3 approved=2 changes_requested=1 commented=0 dismissed=0 other_state=0 inline=0 total=3'
        FROM gold_summary WHERE repo_id = 300 AND activity_date = DATE'2024-02-05'
        UNION ALL
        SELECT '22 repository 300 on 5 February (not touched): people', MAX(people_summary), 'actor_known=2 actors=2 coverage=66.67 name=org/gamma'
        FROM gold_summary WHERE repo_id = 300 AND activity_date = DATE'2024-02-05'
        UNION ALL
        SELECT '23 audit quality_metrics', MAX(metrics_summary), 'repository_days=5 issue_comments=9 on_issues=5 on_prs=3 unclassified=1 reviews=10 state_known=9 approved=4 changes_requested=1 commented=2 dismissed=1 other_state=1 inline=5 total=24 actor_known=20 actors=12'
        FROM audit_summary WHERE run_id = 'test-run3'
        UNION ALL
        SELECT '24 different keys in Gold', CAST(COUNT(DISTINCT repo_id, activity_date) AS STRING), '9' FROM {TEST_GOLD}
    )
    ORDER BY check_name
""")
checks.show(100, truncate=False)

failed_count = checks.filter("result = 'FAIL'").count()
section_results["T11"] = failed_count
if failed_count > 0:
    raise AssertionError(f"T11: {failed_count} check(s) failed. See the table above.")
print("T11: ALL CHECKS PASSED")

## T12. Run 4: removed events

Two Silver rows are removed, from two different tables: the second comment of repository 200 on 2 February (...35) and the only event of repository 500 (...36). Then the source date 2 February runs again.

In [0]:
spark.sql(f"""
    DELETE FROM {TEST_ISSUE_COMMENTS}
    WHERE event_id = '44000000035'
""")

spark.sql(f"""
    DELETE FROM {TEST_REVIEWS}
    WHERE event_id = '44000000036'
""")

print("Issue comment test rows :", spark.table(TEST_ISSUE_COMMENTS).count())
print("Review test rows        :", spark.table(TEST_REVIEWS).count())
print("Review comment test rows:", spark.table(TEST_REVIEW_COMMENTS).count())

In [0]:
%run ../04_repository_collaboration_metrics $test_schema="test_gold_g4" $source_date="2024-02-02" $run_id="test-run4"

## T13. Checks of run 4

Worked out by hand.

- The source date 2 February has 5 rows left: ...20, ...21, ...22, ...23 and the late event ...37. The affected days are still 1 and 2 February.
- 18 rows on 1 February and 4 rows on 2 February: 22 rows, 4 repository-days. By table: 8 issue comments, 9 reviews and 5 inline review comments.
- Repository 200 on 2 February is back to 2 events by one person.
- Repository 500 has no event any more, so its Gold row is gone.
- Gold has 4 + 4 = 8 rows. The rows of 5 February still carry the ID of run 2.
- Totals of the run: 8 issue comments (4 on issues, 3 in pull request conversations, 1 unknown). 9 reviews, 8 with a known state: 3 approved, 1 changes requested, 2 commented, 1 dismissed, 1 other. 5 inline review comments. Total 22. 18 events have an actor. People per row: 7 + 2 + 1 + 0 = 10.

In [0]:
checks = spark.sql(f"""
    SELECT
        check_name,
        actual,
        expected,
        CASE WHEN actual <=> expected THEN 'PASS' ELSE 'FAIL' END AS result
    FROM (
        SELECT '01 audit status' AS check_name, MAX(status) AS actual, 'SUCCEEDED' AS expected
        FROM {TEST_AUDIT} WHERE run_id = 'test-run4'
        UNION ALL
        SELECT '02 audit affected_dates', MAX(CAST(affected_dates AS STRING)), '[2024-02-01, 2024-02-02]'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run4'
        UNION ALL
        SELECT '03 audit Silver rows read', MAX(source_summary), 'issue_comments=8 reviews=9 review_comments=5'
        FROM audit_summary WHERE run_id = 'test-run4'
        UNION ALL
        SELECT '04 audit gold_rows_published', CAST(MAX(gold_rows_published) AS STRING), '4'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run4'
        UNION ALL
        SELECT '05 Gold rows', CAST(COUNT(*) AS STRING), '8' FROM {TEST_GOLD}
        UNION ALL
        SELECT '06 rows of repository 500 (removed)', CAST(count_if(repo_id = 500) AS STRING), '0' FROM {TEST_GOLD}
        UNION ALL
        SELECT '07 repository 200 on 2 February (lowered): events', MAX(event_summary), 'issue_comments=1 on_issues=0 on_prs=1 unclassified=0 reviews=1 state_known=1 approved=1 changes_requested=0 commented=0 dismissed=0 other_state=0 inline=0 total=2'
        FROM gold_summary WHERE repo_id = 200 AND activity_date = DATE'2024-02-02'
        UNION ALL
        SELECT '08 repository 200 on 2 February (lowered): people', MAX(people_summary), 'actor_known=2 actors=1 coverage=100.0 name=org/beta-renamed'
        FROM gold_summary WHERE repo_id = 200 AND activity_date = DATE'2024-02-02'
        UNION ALL
        SELECT '09 repository 200 on 1 February (late event still there): events', MAX(event_summary), 'issue_comments=2 on_issues=2 on_prs=0 unclassified=0 reviews=0 state_known=0 approved=0 changes_requested=0 commented=0 dismissed=0 other_state=0 inline=1 total=3'
        FROM gold_summary WHERE repo_id = 200 AND activity_date = DATE'2024-02-01'
        UNION ALL
        SELECT '10 repository 200 on 1 February (late event still there): people', MAX(people_summary), 'actor_known=3 actors=2 coverage=100.0 name=org/beta'
        FROM gold_summary WHERE repo_id = 200 AND activity_date = DATE'2024-02-01'
        UNION ALL
        SELECT '11 rows of 1 and 2 February written by run 4', CAST(count_if(activity_date <= DATE'2024-02-02' AND _gold_run_id = 'test-run4') AS STRING), '4' FROM {TEST_GOLD}
        UNION ALL
        SELECT '12 rows of 5 February still from run 2', CAST(count_if(activity_date = DATE'2024-02-05' AND _gold_run_id = 'test-run2') AS STRING), '4' FROM {TEST_GOLD}
        UNION ALL
        SELECT '13 audit quality_metrics', MAX(metrics_summary), 'repository_days=4 issue_comments=8 on_issues=4 on_prs=3 unclassified=1 reviews=9 state_known=8 approved=3 changes_requested=1 commented=2 dismissed=1 other_state=1 inline=5 total=22 actor_known=18 actors=10'
        FROM audit_summary WHERE run_id = 'test-run4'
    )
    ORDER BY check_name
""")
checks.show(100, truncate=False)

failed_count = checks.filter("result = 'FAIL'").count()
section_results["T13"] = failed_count
if failed_count > 0:
    raise AssertionError(f"T13: {failed_count} check(s) failed. See the table above.")
print("T13: ALL CHECKS PASSED")

## T14. Run 5: a source date without rows

No Silver row in any of the three tables has the source date 3 February. The run gets `source_date = 2024-02-03`.

In [0]:
%run ../04_repository_collaboration_metrics $test_schema="test_gold_g4" $source_date="2024-02-03" $run_id="test-run5"

## T15. Checks of run 5: `NO_INPUT`, Gold unchanged

There is no affected day, so nothing is published. The Gold table still holds the 4 rows of run 4 and the 4 rows of run 2. The run still writes its audit row.

In [0]:
checks = spark.sql(f"""
    SELECT
        check_name,
        actual,
        expected,
        CASE WHEN actual <=> expected THEN 'PASS' ELSE 'FAIL' END AS result
    FROM (
        SELECT '01 audit status' AS check_name, MAX(status) AS actual, 'NO_INPUT' AS expected
        FROM {TEST_AUDIT} WHERE run_id = 'test-run5'
        UNION ALL
        SELECT '02 audit processing_scope', MAX(processing_scope), 'SINGLE_DATE'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run5'
        UNION ALL
        SELECT '03 audit source_date', CAST(MAX(source_date) AS STRING), '2024-02-03'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run5'
        UNION ALL
        SELECT '04 audit number of affected dates', CAST(MAX(size(affected_dates)) AS STRING), '0'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run5'
        UNION ALL
        SELECT '05 audit Silver rows read', MAX(source_summary), 'issue_comments=0 reviews=0 review_comments=0'
        FROM audit_summary WHERE run_id = 'test-run5'
        UNION ALL
        SELECT '06 audit gold_rows_published', CAST(MAX(gold_rows_published) AS STRING), '0'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run5'
        UNION ALL
        SELECT '07 audit checks_passed', CAST(MAX(checks_passed) AS STRING), 'true'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run5'
        UNION ALL
        SELECT '08 audit error_message', MAX(error_message), CAST(NULL AS STRING)
        FROM {TEST_AUDIT} WHERE run_id = 'test-run5'
        UNION ALL
        SELECT '09 audit repository_days', CAST(MAX(quality_metrics['repository_days']) AS STRING), '0'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run5'
        UNION ALL
        SELECT '10 Gold rows (unchanged)', CAST(COUNT(*) AS STRING), '8' FROM {TEST_GOLD}
        UNION ALL
        SELECT '11 Gold rows written by run 5', CAST(count_if(_gold_run_id = 'test-run5') AS STRING), '0' FROM {TEST_GOLD}
        UNION ALL
        SELECT '12 Gold rows still from run 4', CAST(count_if(_gold_run_id = 'test-run4') AS STRING), '4' FROM {TEST_GOLD}
        UNION ALL
        SELECT '13 Gold rows still from run 2', CAST(count_if(_gold_run_id = 'test-run2') AS STRING), '4' FROM {TEST_GOLD}
    )
    ORDER BY check_name
""")
checks.show(100, truncate=False)

failed_count = checks.filter("result = 'FAIL'").count()
section_results["T15"] = failed_count
if failed_count > 0:
    raise AssertionError(f"T15: {failed_count} check(s) failed. See the table above.")
print("T15: ALL CHECKS PASSED")

## T16. Run 6: three quality gates fail

The Silver notebooks never let such rows into Silver. The test puts three of them into the test tables anyway, all on 5 February:

| Bad row | Table | What is wrong | The gate that must catch it |
|---|---|---|---|
| ...38 | Issue comment | No `repo_id`. Its group has no repository, so the Gold key would be NULL | `KEY_IS_NULL` |
| ...39 | Issue comment | The target `discussion`, in repository 300. Silver writes only `issue`, `pull_request` or nothing. The comment is counted as an issue comment, but in none of the three target columns | `TARGETS_DO_NOT_ADD_UP` |
| ...31 again | Review | A review in repository 310 with the `event_id` of the issue comment ...31. One event ID is now in two tables | `EVENT_ID_NOT_UNIQUE` |

The cell below prepares the faults. The run itself is in the cell after it.

In [0]:
spark.sql("INSERT INTO " + TEST_ISSUE_COMMENTS + """
    (event_id, event_timestamp, event_date, actor_id, repo_id, repo_name,
     comment_action, comment_target_type, _source_date)
VALUES
    -- A row without repo_id
    ('44000000038', TIMESTAMP'2024-02-05 13:00:00', DATE'2024-02-05', 50, NULL, 'org/broken', 'created', 'issue',      DATE'2024-02-05'),
    -- A target that Silver never writes
    ('44000000039', TIMESTAMP'2024-02-05 13:30:00', DATE'2024-02-05', 51, 300,  'org/gamma',  'created', 'discussion', DATE'2024-02-05')
""")

spark.sql("INSERT INTO " + TEST_REVIEWS + """
    (event_id, event_timestamp, event_date, actor_id, repo_id, repo_name,
     review_action, review_state, _source_date)
VALUES
    -- The event ID of the issue comment ...31
    ('44000000031', TIMESTAMP'2024-02-05 14:00:00', DATE'2024-02-05', 52, 310, 'org/delta', 'created', 'approved', DATE'2024-02-05')
""")

print("Issue comment test rows :", spark.table(TEST_ISSUE_COMMENTS).count())
print("Review test rows        :", spark.table(TEST_REVIEWS).count())
print("Review comment test rows:", spark.table(TEST_REVIEW_COMMENTS).count())

### Run 6

**The next cell is EXPECTED to fail** with `RuntimeError: repository_collaboration_metrics FAILED: Quality gates failed: KEY_IS_NULL; TARGETS_DO_NOT_ADD_UP; EVENT_ID_NOT_UNIQUE`. After it has failed, click the cell after it and choose "Run all below".

If this cell runs twice, the audit table gets two rows for run 6. The checks count runs, not rows, so that does no harm.

In [0]:
%run ../04_repository_collaboration_metrics $test_schema="test_gold_g4" $source_date="2024-02-05" $run_id="test-run6-fail"

## T17. Checks of run 6: Gold is unchanged

Worked out by hand.

- The source date 5 February has 14 rows: the 11 good ones and the 3 bad ones. By table: 4 issue comments, 5 reviews and 5 inline review comments. The affected day is 5 February.
- The run staged 5 rows: repositories 300, 310, 320 and 600, and one row without a repository.
- The staged row of repository 300 has 1 issue comment, but its three target columns add up to 0.
- The three gates are named in the order in which the notebook checks them.
- The run must have written its audit row with the status `FAILED` and the names of the gates.
- The Gold table must be exactly what it was: 8 rows, and the rows of 5 February still carry the ID of run 2. The rejected result is only in the staging table.

After the checks, the test removes the three bad rows, so that the next run can succeed.

In [0]:
checks = spark.sql(f"""
    SELECT
        check_name,
        actual,
        expected,
        CASE WHEN actual <=> expected THEN 'PASS' ELSE 'FAIL' END AS result
    FROM (
        SELECT '01 audit status' AS check_name, MAX(status) AS actual, 'FAILED' AS expected
        FROM {TEST_AUDIT} WHERE run_id = 'test-run6-fail'
        UNION ALL
        SELECT '02 audit failed_checks', MAX(failed_checks), 'KEY_IS_NULL; TARGETS_DO_NOT_ADD_UP; EVENT_ID_NOT_UNIQUE'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run6-fail'
        UNION ALL
        SELECT '03 audit error_message', MAX(error_message), 'Quality gates failed: KEY_IS_NULL; TARGETS_DO_NOT_ADD_UP; EVENT_ID_NOT_UNIQUE'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run6-fail'
        UNION ALL
        SELECT '04 audit checks_passed', CAST(MAX(checks_passed) AS STRING), 'false'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run6-fail'
        UNION ALL
        SELECT '05 audit gold_rows_published', CAST(MAX(gold_rows_published) AS STRING), '0'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run6-fail'
        UNION ALL
        SELECT '06 audit affected_dates', MAX(CAST(affected_dates AS STRING)), '[2024-02-05]'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run6-fail'
        UNION ALL
        SELECT '07 audit Silver rows read', MAX(source_summary), 'issue_comments=4 reviews=5 review_comments=5'
        FROM audit_summary WHERE run_id = 'test-run6-fail'
        UNION ALL
        SELECT '08 Gold rows (unchanged)', CAST(COUNT(*) AS STRING), '8' FROM {TEST_GOLD}
        UNION ALL
        SELECT '09 Gold rows written by the failed run', CAST(count_if(_gold_run_id = 'test-run6-fail') AS STRING), '0' FROM {TEST_GOLD}
        UNION ALL
        SELECT '10 Gold rows of 5 February still from run 2', CAST(count_if(activity_date = DATE'2024-02-05' AND _gold_run_id = 'test-run2') AS STRING), '4' FROM {TEST_GOLD}
        UNION ALL
        SELECT '11 Gold rows without repo_id', CAST(count_if(repo_id IS NULL) AS STRING), '0' FROM {TEST_GOLD}
        UNION ALL
        SELECT '12 staging rows of the rejected result', CAST(count_if(_gold_run_id = 'test-run6-fail') AS STRING), '5' FROM {TEST_STAGING}
        UNION ALL
        SELECT '13 staging rows without repo_id', CAST(count_if(repo_id IS NULL) AS STRING), '1' FROM {TEST_STAGING}
        UNION ALL
        SELECT '14 staging row of repository 300: comments and targets',
               MAX(concat('issue_comments=', CAST(issue_comment_events AS STRING),
                          ' targets=', CAST(issue_conversation_comment_events + pr_conversation_comment_events + unclassified_conversation_comment_events AS STRING))),
               'issue_comments=1 targets=0'
        FROM {TEST_STAGING} WHERE repo_id = 300
    )
    ORDER BY check_name
""")
checks.show(100, truncate=False)

failed_count = checks.filter("result = 'FAIL'").count()
section_results["T17"] = failed_count
if failed_count > 0:
    raise AssertionError(f"T17: {failed_count} check(s) failed. See the table above.")
print("T17: ALL CHECKS PASSED")

In [0]:
spark.sql(f"""
    DELETE FROM {TEST_ISSUE_COMMENTS}
    WHERE event_id IN ('44000000038', '44000000039')
""")

# Only the review with the repeated ID is removed. The issue comment ...31 stays.
spark.sql(f"""
    DELETE FROM {TEST_REVIEWS}
    WHERE event_id = '44000000031'
""")

print("Issue comment test rows after the repair :", spark.table(TEST_ISSUE_COMMENTS).count())
print("Review test rows after the repair        :", spark.table(TEST_REVIEWS).count())
print("Review comment test rows after the repair:", spark.table(TEST_REVIEW_COMMENTS).count())

## T18. Run 7: recovery

The same run again, now without the bad rows. The four rows of 5 February are replaced. The four rows of 1 and 2 February are not touched and still carry the ID of run 4.

In [0]:
%run ../04_repository_collaboration_metrics $test_schema="test_gold_g4" $source_date="2024-02-05" $run_id="test-run7"

In [0]:
checks = spark.sql(f"""
    SELECT
        check_name,
        actual,
        expected,
        CASE WHEN actual <=> expected THEN 'PASS' ELSE 'FAIL' END AS result
    FROM (
        SELECT '01 audit status' AS check_name, MAX(status) AS actual, 'SUCCEEDED' AS expected
        FROM {TEST_AUDIT} WHERE run_id = 'test-run7'
        UNION ALL
        SELECT '02 audit gold_rows_published', CAST(MAX(gold_rows_published) AS STRING), '4'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run7'
        UNION ALL
        SELECT '03 audit checks_passed', CAST(MAX(checks_passed) AS STRING), 'true'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run7'
        UNION ALL
        SELECT '04 audit failed_checks', MAX(failed_checks), CAST(NULL AS STRING)
        FROM {TEST_AUDIT} WHERE run_id = 'test-run7'
        UNION ALL
        SELECT '05 Gold rows', CAST(COUNT(*) AS STRING), '8' FROM {TEST_GOLD}
        UNION ALL
        SELECT '06 rows of 5 February written by run 7', CAST(count_if(activity_date = DATE'2024-02-05' AND _gold_run_id = 'test-run7') AS STRING), '4' FROM {TEST_GOLD}
        UNION ALL
        SELECT '07 rows of 1 and 2 February still from run 4', CAST(count_if(activity_date <= DATE'2024-02-02' AND _gold_run_id = 'test-run4') AS STRING), '4' FROM {TEST_GOLD}
        UNION ALL
        SELECT '08 repository 300 on 5 February: events', MAX(event_summary), 'issue_comments=0 on_issues=0 on_prs=0 unclassified=0 reviews=3 state_known=3 approved=2 changes_requested=1 commented=0 dismissed=0 other_state=0 inline=0 total=3'
        FROM gold_summary WHERE repo_id = 300 AND activity_date = DATE'2024-02-05'
        UNION ALL
        SELECT '09 repository 310 on 5 February: events', MAX(event_summary), 'issue_comments=0 on_issues=0 on_prs=0 unclassified=0 reviews=0 state_known=0 approved=0 changes_requested=0 commented=0 dismissed=0 other_state=0 inline=4 total=4'
        FROM gold_summary WHERE repo_id = 310 AND activity_date = DATE'2024-02-05'
        UNION ALL
        SELECT '10 repository 310 on 5 February: people', MAX(people_summary), 'actor_known=4 actors=1 coverage=100.0 name=org/delta'
        FROM gold_summary WHERE repo_id = 310 AND activity_date = DATE'2024-02-05'
        UNION ALL
        SELECT '11 audit Silver rows read', MAX(source_summary), 'issue_comments=2 reviews=4 review_comments=5'
        FROM audit_summary WHERE run_id = 'test-run7'
        UNION ALL
        SELECT '12 different runs in the audit table so far', CAST(COUNT(DISTINCT run_id) AS STRING), '8' FROM {TEST_AUDIT}
    )
    ORDER BY check_name
""")
checks.show(100, truncate=False)

failed_count = checks.filter("result = 'FAIL'").count()
section_results["T18"] = failed_count
if failed_count > 0:
    raise AssertionError(f"T18: {failed_count} check(s) failed. See the table above.")
print("T18: ALL CHECKS PASSED")

## T19. Run 8: a day that no longer exists in Silver

All Silver rows of 5 February are removed from the three tables. A run for the source date 5 February would now find no rows and end as `NO_INPUT`, as in T15, and the four Gold rows of that day would stay. The way to remove such a day is a run for all dates: it replaces the whole table.

In [0]:
spark.sql(f"""
    DELETE FROM {TEST_ISSUE_COMMENTS}
    WHERE event_date = DATE'2024-02-05'
""")

spark.sql(f"""
    DELETE FROM {TEST_REVIEWS}
    WHERE event_date = DATE'2024-02-05'
""")

spark.sql(f"""
    DELETE FROM {TEST_REVIEW_COMMENTS}
    WHERE event_date = DATE'2024-02-05'
""")

print("Issue comment test rows :", spark.table(TEST_ISSUE_COMMENTS).count())
print("Review test rows        :", spark.table(TEST_REVIEWS).count())
print("Review comment test rows:", spark.table(TEST_REVIEW_COMMENTS).count())

In [0]:
%run ../04_repository_collaboration_metrics $test_schema="test_gold_g4" $source_date="" $run_id="test-run8"

## T20. Checks of run 8

Worked out by hand: Silver has 24 rows left. 22 have an event date, on 1 and 2 February: 8 issue comments, 9 reviews and 5 inline review comments. 2 have none. Gold has the 4 rows of those two days and no row of 5 February.

In [0]:
checks = spark.sql(f"""
    SELECT
        check_name,
        actual,
        expected,
        CASE WHEN actual <=> expected THEN 'PASS' ELSE 'FAIL' END AS result
    FROM (
        SELECT '01 audit status' AS check_name, MAX(status) AS actual, 'SUCCEEDED' AS expected
        FROM {TEST_AUDIT} WHERE run_id = 'test-run8'
        UNION ALL
        SELECT '02 audit processing_scope', MAX(processing_scope), 'ALL_DATES'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run8'
        UNION ALL
        SELECT '03 audit affected_dates', MAX(CAST(affected_dates AS STRING)), '[2024-02-01, 2024-02-02]'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run8'
        UNION ALL
        SELECT '04 audit Silver rows read', MAX(source_summary), 'issue_comments=8 reviews=9 review_comments=5'
        FROM audit_summary WHERE run_id = 'test-run8'
        UNION ALL
        SELECT '05 audit rows without an event date', CAST(MAX(rows_excluded['missing_event_date']) AS STRING), '2'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run8'
        UNION ALL
        SELECT '06 audit gold_rows_published', CAST(MAX(gold_rows_published) AS STRING), '4'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run8'
        UNION ALL
        SELECT '07 Gold rows', CAST(COUNT(*) AS STRING), '4' FROM {TEST_GOLD}
        UNION ALL
        SELECT '08 Gold rows of 5 February', CAST(count_if(activity_date = DATE'2024-02-05') AS STRING), '0' FROM {TEST_GOLD}
        UNION ALL
        SELECT '09 rows written by run 8', CAST(count_if(_gold_run_id = 'test-run8') AS STRING), '4' FROM {TEST_GOLD}
    )
    ORDER BY check_name
""")
checks.show(100, truncate=False)

failed_count = checks.filter("result = 'FAIL'").count()
section_results["T20"] = failed_count
if failed_count > 0:
    raise AssertionError(f"T20: {failed_count} check(s) failed. See the table above.")
print("T20: ALL CHECKS PASSED")

## T21. Bad parameter: `source_date`

`2024-2-3` is not in the form `YYYY-MM-DD`. The notebook must stop in its parameter step, before it reads or writes a table.

**The next cell is EXPECTED to fail** with `ValueError: Invalid source_date '2024-2-3'`. After it has failed, click the cell after it and choose "Run all below".

In [0]:
%run ../04_repository_collaboration_metrics $test_schema="test_gold_g4" $source_date="2024-2-3" $run_id="test-bad-date"

### T21 check: where did it stop?

`%run` runs the other notebook inside this one, so the variables of that notebook can be read here. They show how far the stopped run came:

- `source_date` holds the bad value, so step 1.2 has run.
- `run_id` is still the ID of run 8, so the notebook never came to step 1.4, where it reads the run ID. It stopped before it touched a table.

In [0]:
print("source_date:", source_date)
print("run_id     :", run_id)

if source_date != "2024-2-3":
    raise AssertionError("T21: the run with the bad source_date has not run. Run the cell above first.")

if run_id == "test-bad-date":
    raise AssertionError("T21: the notebook did not stop at the bad source_date. It went on to step 1.4.")

section_results["T21"] = 0
print("T21: the notebook stopped at the bad source_date, as expected")

## T22. Bad parameter: `test_schema`

`TEST_WRONG` has capital letters, so it is not a valid test schema name.

**The next cell is EXPECTED to fail** with `ValueError: Invalid test_schema 'TEST_WRONG'`. After it has failed, click the cell after it and choose "Run all below".

In [0]:
%run ../04_repository_collaboration_metrics $test_schema="TEST_WRONG" $source_date="" $run_id="test-bad-schema"

### T22 check: where did it stop?

- `test_schema` holds the bad value, so step 1.3 has run.
- `run_id` is still the ID of run 8, so the notebook never came to step 1.4.

In [0]:
print("test_schema:", test_schema)
print("run_id     :", run_id)

if test_schema != "TEST_WRONG":
    raise AssertionError("T22: the run with the bad test_schema has not run. Run the cell above first.")

if run_id == "test-bad-schema":
    raise AssertionError("T22: the notebook did not stop at the bad test_schema. It went on to step 1.4.")

section_results["T22"] = 0
print("T22: the notebook stopped at the bad test_schema, as expected")

## T23. Bad parameter: `run_id`

The run ID is written into every Gold row, so it may contain only letters, digits, `-` and `_`. `bad.run.id` contains dots.

**The next cell is EXPECTED to fail** with `ValueError: Invalid run_id 'bad.run.id'`. After it has failed, click the cell after it and choose "Run all below".

In [0]:
%run ../04_repository_collaboration_metrics $test_schema="test_gold_g4" $source_date="" $run_id="bad.run.id"

### T23 check: did it run?

- `run_id` holds the bad value, so the notebook came to step 1.4.
- That it stopped there, and wrote nothing, is checked in T24: this run has no audit row and the Gold table is still the one of run 8.

In [0]:
print("test_schema:", test_schema)
print("run_id     :", run_id)

if run_id != "bad.run.id":
    raise AssertionError("T23: the run with the bad run_id has not run. Run the cell above first.")

section_results["T23"] = 0
print("T23: the notebook came to the run_id step. T24 checks that it wrote nothing.")

## T24. Final checks: nothing was written by the three stopped runs, and production is untouched

- The three runs with a bad parameter stopped before the audit step, so the audit table still holds 9 runs and the Gold table is still the one of run 8.
- Every run wrote exactly one audit row. Run 6 is left out of that check, because its cell may have run more than once.
- The versions of the production tables are read again and compared with the versions from T2.

In [0]:
prod_audit_version_after = spark.sql(
    f"DESCRIBE HISTORY {PROD_AUDIT} LIMIT 1"
).collect()[0]["version"]

if spark.catalog.tableExists(PROD_GOLD):
    prod_gold_version_after = spark.sql(
        f"DESCRIBE HISTORY {PROD_GOLD} LIMIT 1"
    ).collect()[0]["version"]
else:
    prod_gold_version_after = -1

if spark.catalog.tableExists(PROD_STAGING):
    prod_staging_version_after = spark.sql(
        f"DESCRIBE HISTORY {PROD_STAGING} LIMIT 1"
    ).collect()[0]["version"]
else:
    prod_staging_version_after = -1

print("Production audit version  :", prod_audit_version_after)
print("Production Gold version   :", prod_gold_version_after)
print("Production staging version:", prod_staging_version_after)

In [0]:
checks = spark.sql(f"""
    SELECT
        check_name,
        actual,
        expected,
        CASE WHEN actual <=> expected THEN 'PASS' ELSE 'FAIL' END AS result
    FROM (
        SELECT '01 different runs in the audit table' AS check_name, CAST(COUNT(DISTINCT run_id) AS STRING) AS actual, '9' AS expected
        FROM {TEST_AUDIT}
        UNION ALL
        SELECT '02 audit rows of the three stopped runs', CAST(count_if(run_id IN ('test-bad-date', 'test-bad-schema', 'bad.run.id')) AS STRING), '0'
        FROM {TEST_AUDIT}
        UNION ALL
        SELECT '03 Gold rows still from run 8', CAST(count_if(_gold_run_id = 'test-run8') AS STRING), '4'
        FROM {TEST_GOLD}
        UNION ALL
        SELECT '04 Gold rows', CAST(COUNT(*) AS STRING), '4'
        FROM {TEST_GOLD}
        UNION ALL
        SELECT '05 runs by status',
               concat('SUCCEEDED=', CAST(COUNT(DISTINCT CASE WHEN status = 'SUCCEEDED' THEN run_id END) AS STRING),
                      ' FAILED=', CAST(COUNT(DISTINCT CASE WHEN status = 'FAILED' THEN run_id END) AS STRING),
                      ' NO_INPUT=', CAST(COUNT(DISTINCT CASE WHEN status = 'NO_INPUT' THEN run_id END) AS STRING)),
               'SUCCEEDED=6 FAILED=1 NO_INPUT=2'
        FROM {TEST_AUDIT}
        UNION ALL
        SELECT '06 production audit table version unchanged', '{prod_audit_version_after}', '{prod_audit_version_before}'
        UNION ALL
        SELECT '07 production Gold table version unchanged', '{prod_gold_version_after}', '{prod_gold_version_before}'
        UNION ALL
        SELECT '08 production staging version unchanged', '{prod_staging_version_after}', '{prod_staging_version_before}'
        UNION ALL
        SELECT '09 audit rows of the 8 runs without run 6 (one row each)', CAST(count_if(run_id <> 'test-run6-fail') AS STRING), '8'
        FROM {TEST_AUDIT}
        UNION ALL
        SELECT '10 audit rows of run 6 are all FAILED', CAST(count_if(run_id = 'test-run6-fail' AND status <> 'FAILED') AS STRING), '0'
        FROM {TEST_AUDIT}
        UNION ALL
        SELECT '11 audit rows with another pipeline or table name', CAST(count_if(pipeline_name <> 'gold_repository_collaboration_metrics' OR gold_table <> '{TEST_GOLD}') AS STRING), '0'
        FROM {TEST_AUDIT}
    )
    ORDER BY check_name
""")
checks.show(100, truncate=False)

failed_count = checks.filter("result = 'FAIL'").count()
section_results["T24"] = failed_count
if failed_count > 0:
    raise AssertionError(f"T24: {failed_count} check(s) failed. See the table above.")
print("T24: ALL CHECKS PASSED")

## Result of the whole test

Every section wrote its number of failed checks into `section_results`. This cell goes through the 13 sections. A section that is not in the dictionary did not run, for example because the cells above it were skipped.

The test has passed only when this cell prints `TEST PASSED`.

In [0]:
EXPECTED_SECTIONS = [
    "T4", "T7", "T9", "T11", "T13", "T15", "T17",
    "T18", "T20", "T21", "T22", "T23", "T24",
]

problems = []

for section in EXPECTED_SECTIONS:
    if section not in section_results:
        problems.append(section + " did not run")
    elif section_results[section] > 0:
        problems.append(section + " has " + str(section_results[section]) + " failed check(s)")

print("Sections that ran:", len(section_results), "of", len(EXPECTED_SECTIONS))

if len(problems) > 0:
    raise AssertionError("TEST FAILED: " + "; ".join(problems))

print("TEST PASSED: all", len(EXPECTED_SECTIONS), "sections ran and every check passed.")

## Clean up: drop the test schema

Only a schema whose name starts with `test_` is ever dropped.

If the test failed, you can leave this cell out and look at the test tables first. The next test run drops the schema anyway, in T1.

In [0]:
if not TEST_SCHEMA.startswith("test_"):
    raise RuntimeError("Refusing to drop a schema that is not a test schema.")

spark.sql(f"DROP SCHEMA IF EXISTS github_lakehouse.{TEST_SCHEMA} CASCADE")

print("Dropped:", TEST_SCHEMA)